In [1]:
import numpy as np
import gurobipy
import pandas as pd

In [10]:
# --- 1. Lecture du fichier ---
df = pd.read_excel("efficacity_data_v1.xlsx")

# ============================
# PARTIE A — CALIBRATED
# ============================

df_calibrated = (
    df[df["id_simulation"].astype(str).str.contains("calibrated", case=False)]
    [["building_name", "id_simulation", "conso_total_mwh_an"]]
)

# Liste des calibrated par bâtiment
calibrated_list = (
    df_calibrated
    .groupby("building_name")["conso_total_mwh_an"]
    .apply(list)
    .to_dict()
)

# ============================
# PARTIE B — SANS CALIBRATED
# ============================

df_nc = df[
    ~df["id_simulation"].astype(str).str.contains("calibrated", case=False)
]

# Colonnes utiles
df_nc = df_nc[[
    "building_name",
    "id_simulation",
    "conso_total_mwh_an",
    "cout_investissement_euros"
]]

# --- 2. Groupement ---
group_conso = df_nc.groupby("building_name")["conso_total_mwh_an"].apply(list)
group_cost  = df_nc.groupby("building_name")["cout_investissement_euros"].apply(list)

# --- 3. Taille maximale ---
max_len = max(
    group_conso.apply(len).max(),
    group_cost.apply(len).max()
)

# --- 4. Matrices à taille fixe ---
conso_matrix = np.array([
    vals + [np.inf] * (max_len - len(vals))
    for vals in group_conso
])

cost_matrix = np.array([
    vals + [np.inf] * (max_len - len(vals))
    for vals in group_cost
])

# --- 5. Métadonnées utiles ---
building_names = group_conso.index.tolist()

print("Conso matrix shape :", conso_matrix.shape)
print("Cost matrix shape  :", cost_matrix.shape)
print("Nb bâtiments       :", len(building_names))
print("Nb calibrated      :", len(df_calibrated))
print(df_calibrated.head())

Conso matrix shape : (71, 15)
Cost matrix shape  : (71, 15)
Nb bâtiments       : 71
Nb calibrated      : 76
                             building_name  \
0                            abri_bouliste   
16  accueil_de_loisirs_maternel_la_varenne   
32             association_31_rue_gambetta   
48               ateliers_municipaux_(ctm)   
59                     base_de_canoe_kayak   

                                        id_simulation  conso_total_mwh_an  
0                        abri_bouliste_ref_calibrated           13.256400  
16  accueil_de_loisirs_maternel_la_varenne_ref_cal...           22.254286  
32         association_31_rue_gambetta_ref_calibrated           12.827150  
48           ateliers_municipaux_(ctm)_ref_calibrated         1148.106350  
59                 base_de_canoe_kayak_ref_calibrated           21.643910  


In [11]:
# Sauvegarde des matrices
np.savetxt(
    "conso_matrix.csv",
    conso_matrix,
    delimiter=",",
    fmt="%.10g"
)

np.savetxt(
    "cost_matrix.csv",
    cost_matrix,
    delimiter=",",
    fmt="%.10g"
)

np.savetxt(
    "calibrated_list.csv",
    np.array(list(calibrated_list.keys()), dtype=str),
    delimiter=",",
    fmt="%s"
)

# Sauvegarde des bâtiments (utile pour l’indexation)
np.savetxt(
    "building_names.csv",
    np.array(building_names, dtype=str),
    delimiter=",",
    fmt="%s"
)